# TikTok Audio Transcript Pipeline

In [2]:
import json, subprocess, time, random, shutil
from pathlib import Path
import pandas as pd

HANDLE = "humbletoker"
PROFILE_URL = f"https://www.tiktok.com/@{HANDLE}"

DATA_DIR = Path("../data")
AUDIO_DIR = DATA_DIR / "audio"
DERIVED_DIR = DATA_DIR / "derived"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

FFMPEG = shutil.which("ffmpeg") or "/Users/Logan/miniforge3/bin/ffmpeg"

SLEEP_MIN, SLEEP_MAX = 0.8, 2.0  # rate-limit friendliness


In [3]:
cmd = ["yt-dlp", "--dump-json", PROFILE_URL]
proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    print(proc.stderr[:4000])
    raise RuntimeError("yt-dlp profile scrape failed.")

rows = []
for line in proc.stdout.splitlines():
    obj = json.loads(line)
    if obj.get("_type") == "playlist":
        continue
    vid = obj.get("id")
    url = obj.get("webpage_url") or obj.get("original_url")
    if not vid or not url:
        continue
    rows.append({
        "video_id": str(vid),
        "video_url": url,
        "title": obj.get("title"),
        "upload_date": obj.get("upload_date"),
        "view_count": obj.get("view_count"),
        "like_count": obj.get("like_count"),
        "comment_count": obj.get("comment_count"),
        "repost_count": obj.get("repost_count"),
    })

videos = (
    pd.DataFrame(rows)
    .drop_duplicates("video_id")
    .sort_values("upload_date", ascending=False)
    .reset_index(drop=True)
)

manifest_path = DERIVED_DIR / "tiktok_manifest.csv"
videos.to_csv(manifest_path, index=False)
print("Videos found:", len(videos))
print("Saved manifest:", manifest_path.resolve())
videos.head()


Videos found: 53
Saved manifest: /Users/Logan/Desktop/TikTok Goodnight/data/derived/tiktok_manifest.csv


,video_id,video_url,title,upload_date,view_count,like_count,comment_count,repost_count
0,7594277449698004238,https://www.tiktok.com/@humbletoker/video/7594...,"Goodnight to everyone EXCEPT challange, level ...",20260112,656,28,3,0
1,7593500470837087501,https://www.tiktok.com/@humbletoker/video/7593...,TikTok video #7593500470837087501,20260109,320,18,3,0
2,7592046457117625613,https://www.tiktok.com/@humbletoker/video/7592...,This may be easier than 2% actually… goodnight...,20260106,1339,53,9,63
3,7542733065643232526,https://www.tiktok.com/@humbletoker/video/7542...,Yeezy SpongeBob hamburger XQC. Comment if you ...,20250826,1965,68,11,72
4,7542350925122293006,https://www.tiktok.com/@humbletoker/video/7542...,"Goodnight to everyone except… Labubu, Africa, ...",20250825,8285,137,31,343


In [4]:
for p in AUDIO_DIR.glob("*"):
    if p.is_file():
        p.unlink()
print("Cleared:", AUDIO_DIR.resolve())


Cleared: /Users/Logan/Desktop/TikTok Goodnight/data/audio


In [5]:
failed_ids = []

for _, r in videos.iterrows():
    vid = r["video_id"]
    url = r["video_url"]
    out_m4a = AUDIO_DIR / f"{vid}.m4a"
    if out_m4a.exists() and out_m4a.stat().st_size > 0:
        continue

    cmd = [
        "yt-dlp", url,
        "--no-playlist",
        "-x",
        "--audio-format", "m4a",
        "--audio-quality", "0",
        "-o", str(AUDIO_DIR / f"{vid}.%(ext)s"),
        "--force-overwrites",
        "--no-warnings",
    ]

    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError:
        failed_ids.append(vid)

    time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))

print("Audio downloaded:", len(list(AUDIO_DIR.glob("*.m4a"))))
print("Initial failures:", len(failed_ids))
failed_ids[:10]



Audio downloaded: 48
Initial failures: 5


['7594277449698004238',
 '7518971343669447991',
 '7518856355789212983',
 '7517490052704783629',
 '7514160603687128366']

In [6]:
still_failed = []

for vid in failed_ids:
    url = f"https://www.tiktok.com/@{HANDLE}/video/{vid}"
    mp4_path = AUDIO_DIR / f"{vid}.mp4"
    m4a_path = AUDIO_DIR / f"{vid}.m4a"

    try:
        # download mp4
        subprocess.run([
            "yt-dlp", url,
            "--no-playlist",
            "-o", str(mp4_path),
            "--force-overwrites",
            "--no-warnings",
        ], check=True, capture_output=True, text=True)

        # extract/convert audio
        subprocess.run([
            FFMPEG, "-y",
            "-i", str(mp4_path),
            "-vn",
            "-ac", "1",
            "-ar", "16000",
            str(m4a_path)
        ], check=True, capture_output=True, text=True)

        mp4_path.unlink(missing_ok=True)

    except subprocess.CalledProcessError:
        still_failed.append(vid)
        mp4_path.unlink(missing_ok=True)

print("After salvage, audio files:", len(list(AUDIO_DIR.glob("*.m4a"))))
print("Still failing:", len(still_failed))
still_failed


After salvage, audio files: 50
Still failing: 3


['7594277449698004238', '7518971343669447991', '7518856355789212983']

In [7]:
from pathlib import Path
import whisper

model = whisper.load_model("base")  # bump to "small" for better accuracy if you want

audio_files = sorted(AUDIO_DIR.glob("*.m4a"))
print("Audio files to transcribe:", len(audio_files))

rows = []
for p in audio_files:
    vid = p.stem
    result = model.transcribe(str(p))
    text = (result.get("text") or "").strip()
    rows.append({"video_id": vid, "transcript": text})

transcripts = pd.DataFrame(rows)

# Save outputs
out_parquet = DERIVED_DIR / "tiktok_transcripts.parquet"
out_csv = DERIVED_DIR / "tiktok_transcripts.csv"
transcripts.to_parquet(out_parquet, index=False)
transcripts.to_csv(out_csv, index=False)

print("Saved:", out_parquet.resolve())
print("Saved:", out_csv.resolve())
transcripts.head()


100%|███████████████████████████████████████| 139M/139M [01:22<00:00, 1.76MiB/s]


Audio files to transcribe: 50


/Users/Logan/miniforge3/lib/python3.12/site-packages/whisper/transcribe.py:133: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/Logan/miniforge3/lib/python3.12/site-packages/whisper/transcribe.py:133: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/Logan/miniforge3/lib/python3.12/site-packages/whisper/transcribe.py:133: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/Logan/miniforge3/lib/python3.12/site-packages/whisper/transcribe.py:133: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/Logan/miniforge3/lib/python3.12/site-packages/whisper/transcribe.py:133: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("F

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.